In [ ]:
!pip install -q pytest

import pytest
from decimal import Decimal, ROUND_HALF_UP

In [ ]:
import os
from decimal import Decimal

cell_content = """
import pytest
from decimal import Decimal, ROUND_HALF_UP

def validate_card(card_number: str) -> bool:
    digits = card_number.replace(' ', '')
    if not digits.isdigit() or not digits:
        return False
    total = 0
    for i, ch in enumerate(reversed(digits)):
        n = int(ch)
        if i % 2 == 1:
            n *= 2
            if n > 9: n -= 9
        total += n
    return total % 10 == 0

def calculate_fee(amount: Decimal, card_type: str) -> Decimal:
    if amount <= 0:
        raise ValueError("Сумма должна быть положительной")
    rates = {'debit': Decimal('0.005'), 'credit': Decimal('0.012'), 'premium': Decimal('0.008')}
    if card_type not in rates:
        raise ValueError(f"Неизвестный тип карты: {card_type}")
    fee = (amount * rates[card_type]).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    return max(Decimal('0.30'), min(Decimal('5.00'), fee))

@pytest.fixture
def sample_cards():
    return ["4111111111111111", "5555555555554444"]

# ТЕСТЫ ГРАНИЧНЫХ УСЛОВИЙ
@pytest.mark.parametrize('card, expected', [
    ('4111 1111 1111 1111', True),
    ('', False),
    ('abc', False),
    ('123', False),
    ('411111111111111', False),
])
def test_validate_card_edges(card, expected):
    assert validate_card(card) == expected

@pytest.mark.parametrize('amt, ctype, expected', [
    (Decimal('100'), 'debit', Decimal('0.50')),
    (Decimal('0.01'), 'debit', Decimal('0.30')),
    (Decimal('10000'), 'debit', Decimal('5.00')),
])
def test_calculate_fee_valid(amt, ctype, expected):
    assert calculate_fee(amt, ctype) == expected

def test_calculate_fee_errors():
    with pytest.raises(ValueError, match="Сумма должна быть положительной"):
        calculate_fee(Decimal('0'), 'debit')
    with pytest.raises(ValueError, match="Сумма должна быть положительной"):
        calculate_fee(Decimal('-10'), 'credit')
    with pytest.raises(ValueError, match="Неизвестный тип карты"):
        calculate_fee(Decimal('100'), 'gold_card')
"""

with open("test_fastpay_final.py", "w") as f:
    f.write(cell_content)

!pytest test_fastpay_final.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.8.5, typeguard-4.5.2, anyio-4.13.0
collected 9 items                                                              

test_fastpay_final.py::test_validate_card_edges[4111 1111 1111 1111-True] PASSED [ 11%]
test_fastpay_final.py::test_validate_card_edges[-False] PASSED           [ 22%]
test_fastpay_final.py::test_validate_card_edges[abc-False] PASSED        [ 33%]
test_fastpay_final.py::test_validate_card_edges[123-False] PASSED        [ 44%]
test_fastpay_final.py::test_validate_card_edges[411111111111111-False] PASSED [ 55%]
test_fastpay_final.py::test_calculate_fee_valid[amt0-debit-expected0] PASSED [ 66%]
test_fastpay_final.py::test_calculate_fee_valid[amt1-debit-expected1] PASSED [ 77%]
test_fastpay_final.py::test_calculate_fee_valid[amt2-debit-expected2] PASSED [

validate_card: Реализован алгоритм Луна с предварительной очисткой строки от пробелов. Тесты проверяют валидные номера, невалидные, пустые строки и некорректные символы.
calculate_fee: Реализован расчет комиссии для трех типов карт (debit, credit, premium) с использованием Decimal для финансовой точности и банковским округлением (ROUND_HALF_UP). Также соблюдены лимиты (минимум 0.3 и максимум 5.0).
Тестирование: Код полностью покрыт параметризованными тестами PyTest, запущенными в изолированном файле для чистоты результата.

Ежели Номер карты =
empty = пустая строка ('') → False.
non-numeric = строка с буквами ('abc') → False.
длина номера = слишком короткие (3 цифры) и некорректные по Луну (15 цифр) номера → False.
Сумма и типы =
сумма 0= выбрасывается ValueError с корректным сообщением.
отрицательная сумма = выбрасывается ValueError.
неизвестный тип карты = передача 'gold_card' приводит к ValueError.
минимум/максимум = суммы в 0.01 и 10000 подтверждают работу лимитов в 0.3 и 5.0 у.е. соответственно.

In [ ]:
from test_fastpay_final import validate_card, calculate_fee
from decimal import Decimal
from datetime import datetime

def log(level, msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{timestamp} - {level:7} - {msg}")

# 10 транзакций
transactions = [
    # УСПЕШНЫЕ (первые 5)
    ("4111111111111111", Decimal("150.00"), "debit"),     # обычный дебет
    ("5555555555554444", Decimal("30.00"),  "credit"),    # кредит
    ("378282246310005",  Decimal("600.00"), "premium"),   # премиум
    ("4111111111111111", Decimal("0.30"),   "debit"),     # дебет, сумма маленькая -> комиссия 0.30
    ("5555555555554444", Decimal("500.00"), "credit"),    # кредит, комиссия урезается до 5.00
    # ОШИБОЧНЫЕ / ГРАНИЧНЫЕ
    ("1234567812345678", Decimal("100.00"), "debit"),     # невалидная карта
    ("4111111111111111", Decimal("-50.00"), "debit"),     # отрицательная сумма
    ("4111111111111111", Decimal("0"),      "debit"),     # нулевая сумма
    ("4111111111111111", Decimal("100.00"), "gold_card"), # неизвестный тип
    ("",                 Decimal("100.00"), "debit"),     # пустой номер
]

log("INFO", "Демонстрация работы модуля на 10 транзакций ")

for idx, (card, amount, card_type) in enumerate(transactions, start=1):
    log("INFO", f"Транзакция #{idx}: карта='{card if card else '(empty)'}', сумма={amount}, тип={card_type}")

    # Проверка карты
    is_valid = validate_card(card)
    log("INFO", f"  -> Валидность карты: {is_valid}")

    if not is_valid:
        log("WARNING", f"  -> Транзакция #{idx} ОТКЛОНЕНА: неверный номер карты")
        continue

    # Расчёт комиссии
    try:
        fee = calculate_fee(amount, card_type)
        total = amount + fee
        log("INFO", f"  -> УСПЕХ: комиссия={fee}, итого={total}")
    except ValueError as e:
        log("ERROR", f"  -> ОШИБКА: {e}")

log("INFO", "Демонстрация завершена")

2026-05-28 01:04:20 - INFO    - Демонстрация работы модуля на 10 транзакций 
2026-05-28 01:04:20 - INFO    - Транзакция #1: карта='4111111111111111', сумма=150.00, тип=debit
2026-05-28 01:04:20 - INFO    -   -> Валидность карты: True
2026-05-28 01:04:20 - INFO    -   -> УСПЕХ: комиссия=0.75, итого=150.75
2026-05-28 01:04:20 - INFO    - Транзакция #2: карта='5555555555554444', сумма=30.00, тип=credit
2026-05-28 01:04:20 - INFO    -   -> Валидность карты: True
2026-05-28 01:04:20 - INFO    -   -> УСПЕХ: комиссия=0.36, итого=30.36
2026-05-28 01:04:20 - INFO    - Транзакция #3: карта='378282246310005', сумма=600.00, тип=premium
2026-05-28 01:04:20 - INFO    -   -> Валидность карты: True
2026-05-28 01:04:20 - INFO    -   -> УСПЕХ: комиссия=4.80, итого=604.80
2026-05-28 01:04:20 - INFO    - Транзакция #4: карта='4111111111111111', сумма=0.30, тип=debit
2026-05-28 01:04:20 - INFO    -   -> Валидность карты: True
2026-05-28 01:04:20 - INFO    -   -> УСПЕХ: комиссия=0.30, итого=0.60
2026-05-28 